### Importing The Required Libraries 

In [1]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict 
from transformers import pipeline 
from langchain.messages import SystemMessage,HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint


C:\Users\rajve\miniconda3\envs\nlp_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Defining The State 

In [14]:
class EmailState(TypedDict):
    email:str
    category:str
    status:str

### Importing The LLM 

In [15]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    task="text-generation",
    max_new_tokens=10,
    temperature=0
)

In [16]:
chat_model=ChatHuggingFace(llm=llm)

### Defining The Nodes 

In [17]:
def categorize(state: EmailState):
    print("Categorizing your email")

    messages = [
        SystemMessage(
            "Classify the email as either sales or billing. "
            "Return ONLY one word: sales or billing."
        ),
        HumanMessage(state["email"])
    ]

    response = chat_model.invoke(messages)
    category = response.content.strip().lower()

    print("LLM Output:", repr(category))

    if "sales" in category:
        return {
            "category": "sales",
            "status": "categorized"
        }

    elif "billing" in category:
        return {
            "category": "billing",
            "status": "categorized"
        }

    else:
        raise ValueError(f"Unexpected LLM output: {category}")

In [26]:
def sales_agent(state:EmailState):
    print('The request has reached the sales department and will be answered soon by our sales team')
    return {"status":"sales-resolved"}

In [27]:
def billing_agent(state:EmailState):
    print('The request has reached the billing department and will be answered soon by our billing team')
    return {"status":"billing-resolved"}

### Making The State Graph 

In [28]:
builder=StateGraph(EmailState)

In [29]:
builder.add_node('categorizer',categorize)
builder.add_node('sales',sales_agent)
builder.add_node('billing',billing_agent)

### Making Router

In [30]:
def router(state:EmailState):
    if state["category"]=="sales":
        return "sales"
    elif state["category"]=="billing":
        return "billing"
    else:
        raise ValueError(f"Unknown category: {state['category']}")

### Adding Edges 

In [31]:
builder.add_edge(START,'categorizer')
builder.add_conditional_edges('categorizer',router)
builder.add_edge('sales',END)
builder.add_edge('billing',END)

### Compiling the Graph

In [32]:
graph=builder.compile()

### Testing The Graph

In [33]:
initial_state=EmailState(
    email=input('Please enter your request or complaint'),
    category='none',
    status='pending'
)
final_state=graph.invoke(initial_state)

Please enter your request or complaint Hi, I would like to purchase 500 units of your enterprise software. What is the pricing?


Categorizing your email
LLM Output: 'sales'
The request has reached the sales department and will be answered soon by our sales team


In [34]:
final_state

{'email': 'Hi, I would like to purchase 500 units of your enterprise software. What is the pricing?',
 'category': 'sales',
 'status': 'sales-resolved'}

In [36]:
    print("\n--- TEST 2: Complaint Email ---")
    email_2 = {"email": "My software keeps crashing every time I open the dashboard. Please help me fix this bug immediately.", "category": "", "status": "New"}
    final_state_2 = graph.invoke(email_2)



--- TEST 2: Complaint Email ---
Categorizing your email
LLM Output: 'billing'
The request has reached the billing department and will be answered soon by our billing team
